In [2]:
!pip install langchain langchain-community langchain-google-genai wikipedia

In [3]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableMap, RunnableLambda

In [4]:
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [9]:
# Set API key
# os.environ["GOOGLE_API_KEY"] = "your_google_api_key"

# Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.25
)

# Wikipedia tool
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Step 1 — Retrieve context
retriever = RunnableLambda(lambda topic: wiki_tool.run(topic))

# Step 2 — Summarization
summary_prompt = ChatPromptTemplate.from_template(
"""
Summarize the following information in 5 sentences.

Information:
{context}
"""
)

summary_chain = summary_prompt | llm

# Step 3 — Quiz generation
quiz_prompt = ChatPromptTemplate.from_template(
"""
Based on the summary below, generate 3 quiz questions.

Summary:
{summary}
"""
)

quiz_chain = quiz_prompt | llm
print(quiz_chain)


first=ChatPromptTemplate(input_variables=['summary'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['summary'], input_types={}, partial_variables={}, template='\nBased on the summary below, generate 3 quiz questions.\n\nSummary:\n{summary}\n'), additional_kwargs={})]) middle=[] last=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', temperature=0.25, client=<google.genai.client.Client object at 0x77fec6090500>, default_metadata=(), model_kwargs={})


In [8]:

# Full pipeline
pipeline = (
    RunnableMap({"context": retriever})
    | summary_chain
    | RunnableLambda(lambda msg: {"summary": msg.content})
    | quiz_chain
)

# Run pipeline
topic = input("Enter a topic: ")

result = pipeline.invoke(topic)

print("\nQuiz Questions:\n")
print(result.content)

Enter a topic: Generative AI

Quiz Questions:

Here are 3 quiz questions based on the summary:

1.  Generative AI (GenAI) is an AI subfield primarily known for its ability to create what?
2.  What specific technological advancements are credited with fueling the recent "AI boom" and the rapid growth of GenAI?
3.  Name two significant concerns or negative impacts associated with the use or development of Generative AI, as mentioned in the summary.
